# 미국 무역데이터 vs 기업 매출 상관관계 분석 V3 — YoY 통일 버전

## V3 변경사항 (V2 대비)
V2는 매출/무역 **레벨(원본값)** 로 상관계수를 계산했는데, 레벨끼리는 둘 다 그냥 우상향 추세를
공유하기만 해도 상관계수가 높게 나오는 착시 문제가 있습니다 (LLY 사례에서 실제로 확인된 문제).

V3는 **YoY 성장률끼리** 상관계수를 계산하도록 수입/수출 양쪽 모두 통일합니다.

### 계산 순서
1. 무역데이터: 월별 패널 → `rolling(3, min_periods=3).sum()`으로 발표주기 무관 트레일링 분기합산
   (V2와 동일) → 그 결과에 `.pct_change(12)`로 YoY 계산 (연속 월별 인덱스라 항상 정확히 12개월 전과 비교됨)
2. 매출: 기업마다 결산일이 들쭉날쭉하고 중간에 리포트가 누락될 수도 있으므로,
   단순히 4분기 앞 값과 비교(`shift(4)`)하지 않고, **각 리포트 시점에서 정확히 1년 전에 가장 가까운
   실제 리포트**(허용오차 ±45일)를 찾아 YoY를 계산합니다. 1년 전 근처에 대응하는 리포트가 없으면
   그 시점은 계산에서 제외됩니다 (분기 하나가 비어도 앞뒤로 전파되는 오류를 방지, 직접 검증 완료).
3. 두 YoY 시계열을 (여전히) 기업의 실제 결산일 기준으로 정렬해 상관계수 계산.

### 성능 최적화 (신규)
티커 하나당 HS 코드 수백 개를 파이썬 반복문 + `pearsonr` 로 하나씩 계산하면 매우 느립니다
(1,962 티커 x 476 HS 코드 = 93만 조합 기준 수십 분 소요). 대신 티커 1명당
**`DataFrame.corrwith()`로 전체 HS 코드를 한 번에 벡터화 계산**하고, p-value도 t분포 공식으로
벡터 연산합니다. scipy.pearsonr과 결과가 완전히 일치하는 것을 확인했고, 150티커x200HS(3만 조합)
기준 12초에 처리 — 실제 스케일에서도 대폭 단축됩니다.

### 유지되는 것
- 매출 소스: `US_IS_from_FMP` (item='revenue', 분기만)
- 발표주기 3그룹(3/6/9/12월, 1/4/7/10월, 2/5/8/11월) 전부 자동 대응
- `ticker`/`hs_code` 조회 인터페이스 동일


In [ ]:

# -*- coding: utf-8 -*-
from __future__ import annotations
import os, sys
from pathlib import Path


def _find_project_root(module_name="DATA", max_up=6):
    here = Path.cwd()
    for base in [here, *list(here.parents)[:max_up]]:
        if (base / module_name).is_dir():
            return base
    return None


try:
    from DATA.stock_invest_function import *
except ModuleNotFoundError:
    _root = _find_project_root("DATA")
    if _root is None:
        raise ModuleNotFoundError("DATA 패키지를 찾을 수 없습니다. 프로젝트 루트에서 실행해 주세요.")
    sys.path.insert(0, str(_root))
    from DATA.stock_invest_function import *
    print(f"[경로 자동 보정] DATA 모듈을 {_root} 에서 찾아 sys.path에 추가했습니다.")

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy import stats
from sqlalchemy import create_engine, text
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

db_info = {"host": get_db_host(), "port": 3307, "user": "stox7412",
           "password": "Apt106503!~", "database": "investar"}


def make_engine(db_info):
    url = (f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
           f"@{db_info['host']}:{int(db_info['port'])}/{db_info['database']}?charset=utf8mb4")
    return create_engine(url, pool_pre_ping=True, pool_recycle=1800,
                          connect_args={"connect_timeout": 10, "read_timeout": 60, "write_timeout": 60})


TABLE_REVENUE = "US_IS_from_FMP"
TABLE_EXPORT_MONTHLY = "us_trade_export_monthly_with_forecast"
TABLE_IMPORT_MONTHLY = "us_trade_import_monthly_with_forecast"


## ⓪ 매출 데이터 자동 백필 (부족하면 FMP에서 자동으로 채워 넣음)

`US_IS_from_FMP`에 티커당 매출 기록이 너무 적으면(예: 최근 8분기만) YoY 상관계수를
계산할 표본이 부족합니다. 아래 셀은 **DB를 먼저 확인해서, 기준치보다 적은 티커만 골라
FMP API에서 자동으로 재수집 + 저장**합니다. 별도로 다른 스크립트를 실행하실 필요 없이
이 노트북만 처음부터 끝까지 실행하시면 됩니다.

- 데이터가 이미 충분한 티커는 건너뜁니다 (불필요한 API 호출 없음)
- 부족한 티커만 최근 `BACKFILL_QUARTERS`개 분기를 다시 받아와 `US_IS_from_FMP`에 UPSERT 저장
- 시간이 좀 걸릴 수 있습니다 (부족한 티커 수 × 약 0.5~0.8초). 진행 상황이 진행바로 표시됩니다.


In [ ]:

import time
import requests
from sqlalchemy import text as _sqltext, bindparam as _bindparam

# ============================================================
# ★ 자동 백필 설정 — 필요하면 이 값들만 조정 ★
# ============================================================
AUTO_BACKFILL_ENABLED = True     # False로 두면 이 단계 전체를 건너뜀
MIN_QUARTERS_REQUIRED = 16       # 이 값 미만인 티커만 백필 대상 (YoY 상관계수용으로 넉넉하게)
BACKFILL_QUARTERS     = 60       # 백필 시 요청할 분기 수 (약 15년치)
FMP_HARD_DEADLINE     = 30       # 티커 하나당 최대 대기시간(초) — 이보다 오래 응답 없으면 강제 포기
FMP_API_KEY           = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
FMP_BASE_URL          = "https://financialmodelingprep.com/api/v3"

_META_COLS = {
    "date", "symbol", "reportedCurrency", "cik",
    "fillingDate", "acceptedDate", "calendarYear",
    "period", "link", "finalLink",
}


from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as _FutureTimeoutError

MAX_CONCURRENT_TICKERS = 20   # 동시에 처리할 티커 수 (FMP 요금제 rate limit 고려해서 조정)
_HTTP_EXECUTOR = ThreadPoolExecutor(max_workers=MAX_CONCURRENT_TICKERS + 10)  # 하드데드라인용 내부 실행기


def _fmp_get(endpoint, params, max_retries=3, timeout=(10, 20), hard_deadline=30):
    """
    timeout=(connect_timeout, read_timeout)은 requests 라이브러리 기준 '조각 사이 최대 공백'이라,
    서버가 아주 느리게 찔끔찔끔 보내면 이론상 총 소요시간이 한없이 길어질 수 있다.
    이를 막기 위해 ThreadPoolExecutor로 요청을 감싸서, hard_deadline(초)을 넘기면
    무조건 포기하고 다음으로 넘어가도록 진짜 하드 데드라인을 건다.
    """
    params = dict(params)
    params["apikey"] = FMP_API_KEY
    url = f"{FMP_BASE_URL}/{endpoint}"

    def _do_request():
        return requests.get(url, params=params, timeout=timeout)

    for attempt in range(max_retries):
        _t0 = time.time()
        try:
            future = _HTTP_EXECUTOR.submit(_do_request)
            resp = future.result(timeout=hard_deadline)
            if resp.status_code == 200:
                data = resp.json()
                return data if isinstance(data, list) else []
            elif resp.status_code == 429:
                time.sleep(20)
            else:
                time.sleep(1)
        except _FutureTimeoutError:
            print(f"    [하드 데드라인 초과] {endpoint} 요청이 {hard_deadline}초 넘게 응답 없음 -> 포기하고 다음으로")
            future.cancel()  # 스레드 자체는 백그라운드에서 계속 돌 수 있으나 결과는 버림
            return []
        except Exception as e:
            elapsed = time.time() - _t0
            print(f"    [재시도 {attempt+1}/{max_retries}] {endpoint} 실패 ({elapsed:.1f}초, {type(e).__name__}) -> 재시도")
            time.sleep(1)
    return []


def _to_long(records, ticker):
    if not records:
        return pd.DataFrame()
    rows = []
    for rec in records:
        date_str = rec.get("date", "")
        period_val = rec.get("period", "")
        for key, val in rec.items():
            if key in _META_COLS:
                continue
            rows.append({"ticker": ticker, "date": date_str, "period": period_val, "item": key, "value": val})
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date_month"] = df["date"].dt.to_period("M").astype(str)
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df


def auto_backfill_shallow_tickers(db_info, min_quarters=MIN_QUARTERS_REQUIRED,
                                   backfill_quarters=BACKFILL_QUARTERS,
                                   max_concurrent=MAX_CONCURRENT_TICKERS, batch_size=200):
    """
    US_IS_from_FMP 에서 revenue 레코드가 min_quarters 미만인 티커만 골라
    FMP에서 backfill_quarters 만큼 재수집 후 UPSERT 저장.
    max_concurrent 개 티커를 동시에 요청해서 전체 소요시간을 단축한다.
    """
    engine = make_engine(db_info)
    with engine.connect() as conn:
        counts = pd.read_sql(_sqltext("""
            SELECT ticker, COUNT(*) AS n
            FROM US_IS_from_FMP
            WHERE item = 'revenue' AND period IN ('Q1','Q2','Q3','Q4')
            GROUP BY ticker
        """), conn)
    engine.dispose()

    shallow_tickers = counts.loc[counts["n"] < min_quarters, "ticker"].tolist()
    print(f"[자동 백필] 전체 {len(counts):,}개 티커 중 {min_quarters}분기 미만: {len(shallow_tickers):,}개")

    if not shallow_tickers:
        print("[자동 백필] 모든 티커가 충분한 데이터를 보유 -> 백필 건너뜀")
        return

    try:
        from US_FMP_FS_2_DB_SAVE_LIB import save_financial_data_incremental
    except ModuleNotFoundError:
        # 이 노트북(analysis 폴더)과 US_FMP_FS_2_DB_SAVE_LIB.py(collect/FMP_data_pipeline 폴더)가
        # 서로 다른 위치에 있을 수 있으므로, 프로젝트 전체를 뒤져서 자동으로 찾아 sys.path에 등록
        _module_dir = None
        _search_root = Path.cwd()
        for _base in [_search_root, *list(_search_root.parents)[:6]]:
            _hits = list(_base.rglob("US_FMP_FS_2_DB_SAVE_LIB.py"))
            if _hits:
                _module_dir = _hits[0].parent
                break
        if _module_dir:
            sys.path.insert(0, str(_module_dir))
            print(f"[경로 자동 보정] US_FMP_FS_2_DB_SAVE_LIB.py 를 '{_module_dir}' 에서 찾아 등록했습니다.")
            try:
                from US_FMP_FS_2_DB_SAVE_LIB import save_financial_data_incremental
            except ModuleNotFoundError as e:
                print("\n" + "!" * 70)
                print("[치명적 오류] 경로를 찾았는데도 US_FMP_FS_2_DB_SAVE_LIB import 실패!")
                print(f"  세부 오류: {e}")
                print("  자동 백필을 건너뜁니다 -> correlation_df가 비어있을 수 있습니다.")
                print("!" * 70 + "\n")
                return
        else:
            print("\n" + "!" * 70)
            print("[치명적 오류] US_FMP_FS_2_DB_SAVE_LIB.py 파일을 프로젝트 어디서도 찾지 못했습니다!")
            print("  자동 백필을 건너뜁니다 -> 이 상태로 진행하면 correlation_df가 비어있을 수 있습니다.")
            print("!" * 70 + "\n")
            return

    is_buffer = []
    n_saved_total = 0
    n_empty = 0
    n_done = 0
    batch_size = min(batch_size, 300)  # 너무 자주 저장하면 매번 ensure_unique_index()가 돌아 부하/락 경합 위험 증가

    def _flush():
        nonlocal is_buffer, n_saved_total
        if not is_buffer:
            return
        IS = pd.concat(is_buffer, ignore_index=True)
        for attempt in range(3):
            try:
                save_financial_data_incremental(IS=IS, mode="upsert", verify=False, verbose=False)
                n_saved_total += len(IS)
                is_buffer = []
                return
            except Exception as e:
                print(f"    [저장 실패, 재시도 {attempt+1}/3] {type(e).__name__}: {str(e)[:150]}")
                time.sleep(10)
        print(f"    [경고] 이 배치({len(IS):,}행)는 3번 재시도해도 저장 실패 -> 이번 배치는 건너뛰고 계속 진행")
        is_buffer = []

    def _fetch_one(ticker):
        """워커 스레드에서 실행 — 네트워크 요청만 하고 DB 저장은 메인 스레드에서 처리."""
        records = _fmp_get(f"income-statement/{ticker}",
                            {"period": "quarter", "limit": backfill_quarters})
        return ticker, records

    interrupted = False
    outer_pool = ThreadPoolExecutor(max_workers=max_concurrent)
    try:
        with tqdm(total=len(shallow_tickers), desc="FMP 매출 백필", unit="ticker") as pbar:
            futures = {outer_pool.submit(_fetch_one, t): t for t in shallow_tickers}
            for future in as_completed(futures):
                ticker = futures[future]
                try:
                    _, records = future.result()
                except Exception as e:
                    records = []
                    print(f"    [실패] {ticker}: {type(e).__name__}")

                if records:
                    long_df = _to_long(records, ticker)
                    if not long_df.empty:
                        is_buffer.append(long_df)
                    else:
                        n_empty += 1
                else:
                    n_empty += 1

                n_done += 1
                pbar.update(1)

                if len(is_buffer) >= batch_size:
                    _flush()
    except KeyboardInterrupt:
        interrupted = True
        print("\n[중단 감지] 사용자가 중단했습니다 -> 지금까지 받은 데이터는 저장하고 종료합니다.")
        outer_pool.shutdown(wait=False, cancel_futures=True)
    finally:
        _flush()  # 정상 종료든 중단이든 마지막 남은 버퍼는 항상 저장
        outer_pool.shutdown(wait=False, cancel_futures=True)

    if interrupted:
        print(f"[자동 백필 중단됨] 약 {n_saved_total:,} 행 저장(UPSERT) | FMP 응답 없음/빈 응답: {n_empty:,}개 티커")
        print("       (전체를 다 돌리진 못했지만, 다음에 다시 실행하면 이미 채워진 티커는 자동으로 건너뜁니다)")
        return

    print(f"[자동 백필 완료] 약 {n_saved_total:,} 행 저장(UPSERT) | FMP 응답 없음/빈 응답: {n_empty:,}개 티커")

    # 백필 후 재확인 (실제로 늘었는지 즉시 검증)
    engine2 = make_engine(db_info)
    stmt = _sqltext("""
        SELECT ticker, COUNT(*) AS n
        FROM US_IS_from_FMP
        WHERE item = 'revenue' AND period IN ('Q1','Q2','Q3','Q4')
          AND ticker IN :tickers
        GROUP BY ticker
    """).bindparams(_bindparam("tickers", expanding=True))
    with engine2.connect() as conn:
        counts_after = pd.read_sql(stmt, conn, params={"tickers": shallow_tickers})
    engine2.dispose()
    still_shallow = int((counts_after["n"] < min_quarters).sum()) if not counts_after.empty else len(shallow_tickers)
    print(f"[재확인] 백필 대상 {len(shallow_tickers):,}개 중 여전히 {min_quarters}분기 미만: {still_shallow:,}개")


def preflight_fix_unique_index(db_info, table_name="US_IS_from_FMP",
                                key_columns=("ticker", "period", "date_month", "item")):
    """
    US_FMP_FS_2_DB_SAVE_LIB.ensure_unique_index() 의 전체 테이블 자기조인 DELETE는
    테이블이 커질수록 락 타임아웃이 반복되며 영원히 실패할 수 있다 (경쟁 트랜잭션이 없어도,
    자기조인 자체가 테이블이 클수록 오래 걸려서 innodb_lock_wait_timeout을 넘길 수 있음).

    이를 막기 위해 티커 단위로 쪼개서(=한 번에 훨씬 적은 행만 잠금) 미리 중복을 제거하고
    유니크 인덱스를 직접 만들어둔다. 이 작업이 한 번 성공하면, 이후 save_financial_data_incremental
    호출 시 ensure_unique_index()는 이미 인덱스가 있는 것으로 보고 그냥 넘어간다.
    """
    engine = make_engine(db_info)
    idx_name = f"uniq_{table_name}_dedup"

    with engine.connect() as conn:
        conn.execute(_sqltext("SET SESSION innodb_lock_wait_timeout = 600"))

        existing = conn.execute(
            _sqltext(f"SHOW INDEX FROM `{table_name}` WHERE Key_name = :idx"), {"idx": idx_name}
        ).fetchall()
        if existing:
            print(f"[프리플라이트] 이미 유니크 인덱스({idx_name})가 있습니다 -> 건너뜀")
            engine.dispose()
            return

        cols = [r[0] for r in conn.execute(_sqltext(f"SHOW COLUMNS FROM `{table_name}`")).fetchall()]
        if "id" not in cols:
            conn.execute(_sqltext(
                f"ALTER TABLE `{table_name}` ADD COLUMN `id` BIGINT UNSIGNED NOT NULL AUTO_INCREMENT PRIMARY KEY FIRST"
            ))
            conn.commit()

        tickers = [r[0] for r in conn.execute(_sqltext(f"SELECT DISTINCT ticker FROM `{table_name}`")).fetchall()]
        join_cond = " AND ".join(f"t1.`{c}` <=> t2.`{c}`" for c in key_columns)

        print(f"[프리플라이트] {len(tickers):,}개 티커 단위로 나눠서 중복 제거 중 "
              f"(테이블 전체를 한 번에 잠그지 않아 락 위험이 훨씬 낮습니다)...")
        n_removed = 0
        for ticker in tqdm(tickers, desc="티커별 중복제거"):
            del_sql = _sqltext(f"""
                DELETE t1 FROM `{table_name}` t1
                INNER JOIN `{table_name}` t2
                ON {join_cond} AND t1.id < t2.id
                WHERE t1.ticker = :ticker
            """)
            try:
                result = conn.execute(del_sql, {"ticker": ticker})
                conn.commit()
                n_removed += result.rowcount or 0
            except Exception as e:
                print(f"  [건너뜀] {ticker}: {str(e)[:120]}")

        print(f"[프리플라이트] 중복 {n_removed:,}행 제거 완료. 유니크 인덱스 생성 시도...")
        cols_str = ", ".join(f"`{c}`" for c in key_columns)
        try:
            conn.execute(_sqltext(f"ALTER TABLE `{table_name}` ADD UNIQUE INDEX `{idx_name}` ({cols_str})"))
            conn.commit()
            print(f"[프리플라이트 완료] 유니크 인덱스 생성 성공 -> 이후 저장에서 이 문제 재발하지 않습니다.")
        except Exception as e:
            print(f"[프리플라이트] 인덱스 생성 실패: {str(e)[:200]}")
    engine.dispose()


if AUTO_BACKFILL_ENABLED:
    preflight_fix_unique_index(db_info)
    auto_backfill_shallow_tickers(db_info)
else:
    print("[자동 백필] AUTO_BACKFILL_ENABLED=False -> 건너뜀")


In [ ]:

# ============================================================
# 데이터 로딩 헬퍼
# ============================================================

def _month_end(ts):
    return pd.Timestamp(ts) + pd.offsets.MonthEnd(0)


def _detect_version_col(conn, table):
    res = conn.execute(text(f"SHOW COLUMNS FROM {table}"))
    cols = [r[0] for r in res.fetchall()]
    for c in ("created_at", "input_date"):
        if c in cols:
            return c
    return None


def load_fmp_quarterly_revenue(engine, table=TABLE_REVENUE, start_date=None, end_date=None):
    """
    US_IS_from_FMP 에서 분기 매출(item='revenue', period in Q1~Q4)만 로드.
    반환: DataFrame[ticker, date(결산일, 월말로 정규화), revenue]
    """
    date_filter = ""
    params = {}
    if start_date:
        date_filter += " AND date >= :start_date"
        params["start_date"] = start_date
    if end_date:
        date_filter += " AND date <= :end_date"
        params["end_date"] = end_date

    sql = text(f"""
        SELECT ticker, date, value AS revenue
        FROM {table}
        WHERE item = 'revenue'
          AND period IN ('Q1','Q2','Q3','Q4')
          AND value IS NOT NULL AND value > 0
          {date_filter}
        ORDER BY ticker, date
    """)
    with engine.connect() as conn:
        rows = conn.execute(sql, params).fetchall()
    df = pd.DataFrame(rows, columns=["ticker", "date", "revenue"])
    if df.empty:
        raise ValueError(f"{table} 에서 분기 매출 데이터를 찾지 못했습니다.")
    df["date"] = pd.to_datetime(df["date"]).apply(_month_end)
    # 같은 (ticker, date) 중복 방지 (여러 period 표기가 겹치는 경우 방어적으로 마지막 값 사용)
    df = df.drop_duplicates(subset=["ticker", "date"], keep="last")
    return df


def load_trade_monthly_panel(engine, direction="import", start_date=None):
    """
    HS 코드별 월별 실측(forecast_flag/is_forecast==0) 무역 데이터를
    wide 패널(index=월말, columns=hs_code)로 로드.
    """
    if direction == "import":
        table = TABLE_IMPORT_MONTHLY
        sql = text(f"""
            SELECT hs_code_6d AS hs_code, date, impDlr AS value
            FROM {table}
            WHERE forecast_flag = 0 AND impDlr IS NOT NULL
            {"AND date >= :start_date" if start_date else ""}
        """)
    elif direction == "export":
        table = TABLE_EXPORT_MONTHLY
        with engine.connect() as conn:
            vcol = _detect_version_col(conn, table)
        vcol_join = f"""
            JOIN (SELECT hs_code, MAX({vcol}) AS mx FROM {table} GROUP BY hs_code) l
              ON t.hs_code = l.hs_code AND t.{vcol} = l.mx
        """ if vcol else ""
        sql = text(f"""
            SELECT t.hs_code AS hs_code, t.date_month_end AS date, t.expDlr AS value
            FROM {table} t
            {vcol_join}
            WHERE t.is_forecast = 0 AND t.expDlr IS NOT NULL
            {"AND t.date_month_end >= :start_date" if start_date else ""}
        """)
    else:
        raise ValueError("direction must be 'import' or 'export'")

    params = {"start_date": start_date} if start_date else {}
    with engine.connect() as conn:
        rows = conn.execute(sql, params).fetchall()
    df = pd.DataFrame(rows, columns=["hs_code", "date", "value"])
    if df.empty:
        raise ValueError(f"{table} 에서 실측 데이터를 찾지 못했습니다.")
    df["date"] = pd.to_datetime(df["date"]).apply(_month_end)
    df["hs_code"] = df["hs_code"].astype(str).str.zfill(6)

    panel = df.pivot_table(index="date", columns="hs_code", values="value", aggfunc="sum").sort_index()
    # 월별 패널을 완전한 연속 월말 인덱스로 재색인 (중간 결측월 방지)
    full_index = pd.date_range(panel.index.min(), panel.index.max(), freq="ME")
    return panel.reindex(full_index)


def build_trailing_quarter_panel(monthly_panel: pd.DataFrame) -> pd.DataFrame:
    """
    월별 패널에 rolling(3, min_periods=3).sum() 적용.
    이 결과의 각 행(월말 date)은 '그 달을 포함한 직전 3개월 합계'이므로,
    어떤 결산주기(3/6/9/12월, 1/4/7/10월, 2/5/8/11월)의 기업이든
    자신의 실제 결산일 그대로 이 패널에서 값을 꺼내 쓰면 정확히 일치한다.
    """
    return monthly_panel.rolling(3, min_periods=3).sum()


def compute_trade_yoy_panel(trailing_quarter_panel: pd.DataFrame) -> pd.DataFrame:
    """
    트레일링 분기합산 패널(연속 월별 인덱스)에 YoY(전년동기대비) 적용.
    인덱스가 연속 월별이라 12개월 시프트가 항상 정확히 '1년 전 같은 트레일링 분기'와 비교된다.
    """
    return trailing_quarter_panel.pct_change(12) * 100.0


def compute_revenue_yoy(series: pd.Series, tol_days: int = 45) -> pd.Series:
    """
    기업 매출 시계열(index=결산일, 들쭉날쭉/결측 가능)의 YoY 계산.
    각 리포트 시점에서 정확히 1년 전(±tol_days 이내)에 가장 가까운 실제 리포트를 찾아 비교.
    1년 전 근처에 대응하는 리포트가 없으면 그 시점은 결과에서 제외된다.
    """
    series = series.sort_index()
    idx = series.index
    target_dates = idx - pd.DateOffset(years=1)
    base_vals = series.reindex(target_dates, method="nearest", tolerance=pd.Timedelta(days=tol_days))
    base_vals.index = idx
    with np.errstate(invalid="ignore", divide="ignore"):
        yoy = (series.values / base_vals.values - 1.0) * 100.0
    return pd.Series(yoy, index=idx).replace([np.inf, -np.inf], np.nan).dropna()


def summarize_fiscal_cycles(revenue_df: pd.DataFrame) -> pd.Series:
    """티커별 결산월(month%3) 분포를 확인 (참고/진단용)."""
    tmp = revenue_df.copy()
    tmp["cycle_group"] = tmp["date"].dt.month % 3
    per_ticker_group = tmp.groupby("ticker")["cycle_group"].agg(lambda s: s.mode().iloc[0])
    label_map = {0: "표준(3/6/9/12월)", 1: "A그룹(1/4/7/10월)", 2: "B그룹(2/5/8/11월)"}
    return per_ticker_group.map(label_map).value_counts()


In [ ]:

# ============================================================
# (선택) 12개월 향후 성장률 상위 HS 코드 필터링용 헬퍼
# ============================================================

def get_top_growth_hs_codes(engine, direction="import", top_n=30, explosive_threshold=500.0):
    monthly_panel = load_trade_monthly_panel(engine, direction=direction)
    base_date = monthly_panel.index.max()
    past_start = _month_end(base_date - pd.DateOffset(months=11))

    rows = []
    for hs in monthly_panel.columns:
        s = monthly_panel[hs]
        past = s.loc[past_start:base_date].sum()
        # 향후 12개월 실측치는 아직 없으므로(참고용), 여기서는 과거 성장 추세만 정보로 제공하고
        # 실제 향후 성장률 랭킹은 기존 build_export_growth_long / build_import_growth_long 결과를
        # CSV로 저장해 두셨다면 그걸 불러와 필터링하는 것을 권장합니다.
        rows.append(dict(hs_code=hs, past_12m_sum=past))
    return pd.DataFrame(rows)


def load_hs_code_filter_from_csv(csv_path, hs_code_col="hs_code"):
    """이전에 저장해둔 '향후 12개월 성장률 랭킹' CSV에서 HS 코드 목록만 추출."""
    df = pd.read_csv(csv_path)
    return df[hs_code_col].astype(str).str.zfill(6).unique().tolist()


In [ ]:

# ============================================================
# 메인 함수: 매출 vs 무역데이터 상관계수 (발표주기 반영)
# ============================================================

def calculate_revenue_trade_correlation(db_info, direction="import",
                                         start_date="2015-01-01", end_date=None,
                                         hs_code_filter=None, ticker_filter=None,
                                         min_periods=8, save_path=None):
    """
    기업 분기 매출과 HS 코드별 무역데이터의 상관계수 분석 (발표주기 불일치 문제 해결판).

    Parameters
    ----------
    direction : 'import' 또는 'export'
    hs_code_filter : list[str] or None
        None이면 전체 HS 코드 대상. 리스트를 주면 해당 HS 코드만 대상으로 계산
        (예: 향후 12개월 성장률 상위 30개로 제한하고 싶을 때 사용)
    ticker_filter : list[str] or None
        None이면 전체 티커 대상.

    Returns
    -------
    pd.DataFrame[ticker, hs_code, correlation, p_value, n_periods,
                 first_quarter, last_quarter, fiscal_cycle]
    """
    engine = make_engine(db_info)

    print("=" * 80)
    print(f"기업 매출 YoY vs HS 코드 {direction} YoY 상관관계 분석 (발표주기 반영, V3)")
    print("=" * 80)

    print("\n[Step 1] 분기 매출 로드 (US_IS_from_FMP)")
    revenue_df = load_fmp_quarterly_revenue(engine, start_date=start_date, end_date=end_date)
    if ticker_filter:
        revenue_df = revenue_df[revenue_df["ticker"].isin(ticker_filter)]
    print(f"  기업 수: {revenue_df['ticker'].nunique():,} / 레코드: {len(revenue_df):,}")

    cycle_summary = summarize_fiscal_cycles(revenue_df)
    print("\n  [결산주기 분포]")
    for label, cnt in cycle_summary.items():
        print(f"    {label}: {cnt:,}개사")

    print(f"\n[Step 2] 월별 {direction} 무역데이터 로드 + 트레일링 3개월 합계 패널 생성")
    monthly_panel = load_trade_monthly_panel(engine, direction=direction, start_date=start_date)
    if hs_code_filter:
        keep_cols = [c for c in monthly_panel.columns if c in set(hs_code_filter)]
        monthly_panel = monthly_panel[keep_cols]
    trailing_q_panel = build_trailing_quarter_panel(monthly_panel)
    trade_yoy_panel = compute_trade_yoy_panel(trailing_q_panel)   # <- V3: YoY로 변환 (연속 월별 인덱스라 안전)
    print(f"  HS 코드 수: {trailing_q_panel.shape[1]:,} / 월말 시점 수: {trailing_q_panel.shape[0]:,}")

    engine.dispose()

    print("\n[Step 3] 티커별 매출 YoY 시계열 구성 (자신의 실제 결산일 기준, 1년전 리포트 최근접 매칭)")
    revenue_pivot = revenue_df.pivot_table(index="date", columns="ticker", values="revenue", aggfunc="last")

    tickers = revenue_pivot.columns.tolist()
    hs_codes = trade_yoy_panel.columns.tolist()
    total = len(tickers) * len(hs_codes)
    print(f"  총 조합 수: {len(tickers):,} tickers x {len(hs_codes):,} HS codes = {total:,}")

    print("\n[Step 4] 상관계수 계산 (YoY 성장률 기준, 티커별 전체 HS 코드 벡터화)")
    results = []
    with tqdm(total=len(tickers), desc="티커 처리", unit="ticker") as pbar:
        for ticker in tickers:
            rev_series = revenue_pivot[ticker].dropna()
            rev_yoy = compute_revenue_yoy(rev_series, tol_days=45)   # <- V3: 레벨 대신 YoY
            if rev_yoy.empty or len(rev_yoy) < min_periods:
                pbar.update(1)
                continue

            # 이 티커의 실제 결산일(YoY 계산된 시점)들을 그대로 트레일링 분기 YoY 패널에서 조회
            trade_aligned_all = trade_yoy_panel.reindex(rev_yoy.index)

            # --- 벡터화: 476개(또는 전체) HS 코드를 한 번에 상관계수 계산 ---
            corr_vec = trade_aligned_all.corrwith(rev_yoy)  # pandas 내장 벡터화 상관계수
            valid_mask = trade_aligned_all.notna() & rev_yoy.notna().values[:, None]
            n_obs = valid_mask.sum(axis=0)

            keep = n_obs[n_obs >= min_periods].index
            if len(keep) == 0:
                pbar.update(1)
                continue

            r_arr = corr_vec.loc[keep].values
            n_arr = n_obs.loc[keep].values.astype(float)
            with np.errstate(invalid="ignore", divide="ignore"):
                t_arr = r_arr * np.sqrt(np.clip(n_arr - 2, 0, None)) / np.sqrt(np.clip(1 - r_arr ** 2, 1e-12, None))
                p_arr = 2 * (1 - stats.t.cdf(np.abs(t_arr), np.clip(n_arr - 2, 1, None)))

            valid_r = ~np.isnan(r_arr)
            for hs_code, r_val, p_val, n_val in zip(
                np.array(keep)[valid_r], r_arr[valid_r], p_arr[valid_r], n_arr[valid_r]
            ):
                mask_dates = trade_aligned_all[hs_code].notna() & rev_yoy.notna()
                common_idx = rev_yoy.index[mask_dates.values]
                results.append(dict(
                    ticker=ticker, hs_code=hs_code, correlation=float(r_val), p_value=float(p_val),
                    n_periods=int(n_val),
                    first_quarter=common_idx.min().strftime("%Y-%m-%d"),
                    last_quarter=common_idx.max().strftime("%Y-%m-%d"),
                ))
            pbar.update(1)

    if not results:
        print("\n[경고] 유효한 상관계수가 없습니다.")
        return pd.DataFrame()

    result_df = pd.DataFrame(results)
    result_df["abs_correlation"] = result_df["correlation"].abs()
    result_df = result_df.sort_values("abs_correlation", ascending=False).drop(columns="abs_correlation").reset_index(drop=True)

    print(f"\n[완료] 유효 상관계수 {len(result_df):,}건")
    print(f"  평균 {result_df['correlation'].mean():.4f} / 중앙값 {result_df['correlation'].median():.4f}")

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        fname = f"revenue_{direction}_correlation_v2_{pd.Timestamp.today():%Y%m%d}.csv"
        fp = os.path.join(save_path, fname)
        result_df.to_csv(fp, index=False, encoding="utf-8-sig")
        print(f"\n[저장] {fp} ({len(result_df):,} rows)")

    return result_df


In [ ]:

# ============================================================
# 조회 인터페이스: ticker -> HS코드 랭킹 / hs_code -> ticker 랭킹
# ============================================================

def get_top_hscodes_for_ticker(correlation_df, ticker, n=10, min_abs_corr=None):
    """특정 ticker에 대해 상관계수 절댓값 기준 상위 n개 HS 코드 반환."""
    df = correlation_df[correlation_df["ticker"] == ticker].copy()
    if min_abs_corr:
        df = df[df["correlation"].abs() >= min_abs_corr]
    df["abs_correlation"] = df["correlation"].abs()
    return (df.sort_values("abs_correlation", ascending=False)
              .drop(columns="abs_correlation")
              .head(n)
              .reset_index(drop=True))


def get_top_tickers_for_hscode(correlation_df, hs_code, n=10, min_abs_corr=None):
    """특정 hs_code에 대해 상관계수 절댓값 기준 상위 n개 ticker 반환."""
    hs_code = str(hs_code).zfill(6)
    df = correlation_df[correlation_df["hs_code"] == hs_code].copy()
    if min_abs_corr:
        df = df[df["correlation"].abs() >= min_abs_corr]
    df["abs_correlation"] = df["correlation"].abs()
    return (df.sort_values("abs_correlation", ascending=False)
              .drop(columns="abs_correlation")
              .head(n)
              .reset_index(drop=True))


def get_top_correlations(correlation_df, ticker=None, hs_code=None, top_n=10, min_correlation=None):
    """(V1 호환용) ticker/hs_code 둘 다 또는 둘 중 하나만 지정해 필터링."""
    df = correlation_df.copy()
    if ticker:
        df = df[df["ticker"] == ticker]
    if hs_code:
        df = df[df["hs_code"] == str(hs_code).zfill(6)]
    if min_correlation:
        df = df[df["correlation"].abs() >= min_correlation]
    df["abs_correlation"] = df["correlation"].abs()
    return df.sort_values("abs_correlation", ascending=False).drop(columns="abs_correlation").head(top_n).reset_index(drop=True)


## ① 실행

In [ ]:

save_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\US_trade_revenue_corr"

# (선택) 향후 12개월 성장률 상위 HS 코드로 제한하고 싶으면,
# 이전에 저장한 랭킹 CSV에서 코드 목록을 불러와 hs_code_filter로 넘기세요.
# hs_code_filter = load_hs_code_filter_from_csv(r"...\US_import_growth_ranking.csv")
hs_code_filter = None   # None = 전체 HS 코드 대상

correlation_df = calculate_revenue_trade_correlation(
    db_info=db_info,
    direction="import",          # 'import' 또는 'export'
    start_date="2015-01-01",
    end_date=None,
    hs_code_filter=hs_code_filter,
    min_periods=8,
    save_path=save_path,
)


## ② 조회 예시

In [ ]:

# 특정 ticker -> 상관계수 높은 순 HS 코드 n개
ticker_result = get_top_hscodes_for_ticker(correlation_df, ticker="AAPL", n=15)
print("[AAPL] 상관계수 상위 HS 코드")
print(ticker_result.to_string(index=False))

print()

# 특정 hs_code -> 상관계수 높은 순 ticker n개
hs_result = get_top_tickers_for_hscode(correlation_df, hs_code="854232", n=15)
print("[854232] 상관계수 상위 ticker")
print(hs_result.to_string(index=False))
